# Stage 2: Algerian Darija <-> English translation fine-tuning
Continues fine-tuning the Stage 1 LoRA adapter on bidirectional translation, using explicit translation instructions in every example so the model gets used to acting as a translation assistant.

In [ ]:
!pip install -q \
    "transformers==5.16.1" \
    "datasets" \
    "peft" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "huggingface_hub" \
    "scikit-learn"

In [ ]:
import transformers

print(transformers.__version__)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoConfig, BitsAndBytesConfig, AutoModelForCausalLM, TextStreamer, TrainingArguments, Trainer
from huggingface_hub import login
from sklearn.model_selection import train_test_split
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModelForCausalLM
)
from trl import SFTTrainer

In [ ]:
# login to Hugging Face using the token from the .env file
# from dotenv import load_dotenv
# load_dotenv()
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("Warning: HF_TOKEN not found in environment")
model_id = "google/gemma-4-E2B"
stage1_adapter_dir = "/kaggle/input/datasets/YOUR_USERNAME/gemma4-darija-qlora"  # path to your Stage 1 adapter dataset
device_map = {"": 0}
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

In [ ]:
# Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["vision_tower", "audio_tower"],
)

In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, extra_special_tokens={})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

streamer = TextStreamer(tokenizer, skip_prompt=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={'': 0},
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    attn_implementation="sdpa",  # Accelerated PyTorch Scaled Dot-Product Attention
)

In [ ]:
print("Model loaded successfully. Skipping full model print to save memory.")

In [ ]:
# Bypass prepare_model_for_kbit_training to avoid OOM on Gemma4's
# embed_tokens_per_layer (262144 x 8960 bf16 = ~4.7GB -> fp32 = ~9.4GB -> OOM)

# Step 1: Disable KV cache (required for gradient checkpointing)
model.config.use_cache = False

# Step 2: Enable gradient checkpointing directly
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Step 3: Selectively upcast small non-4bit params to float32
# (skip large embeddings that would exceed 8GB VRAM)
SKIP_UPCAST = {"embed_tokens_per_layer", "embed_tokens", "lm_head"}
for name, param in model.named_parameters():
    if param.__class__.__name__ == "Params4bit":
        continue  # quantized weights - leave as-is
    if param.dtype not in (torch.float16, torch.bfloat16):
        continue  # already float32 or other dtype
    if any(skip in name for skip in SKIP_UPCAST):
        continue  # too large to safely upcast on 8GB VRAM
    param.data = param.data.to(torch.float32)

# Step 4: Freeze all base-model parameters
for param in model.parameters():
    param.requires_grad_(False)

import gc
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after prep: {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

# Load the Stage 1 adapter and keep training it (instead of creating a fresh LoRA)
model = PeftModelForCausalLM.from_pretrained(model, stage1_adapter_dir, is_trainable=True)
model.print_trainable_parameters()

## Build the bidirectional translation dataset
Each CSV row (`id, Arabic, English`) produces **two** training examples: Darija -> English and English -> Darija. The instruction is always written in the same language as the input, and it stays identical to what we'll use at inference time.

In [ ]:
EN_TO_AR_TEMPLATE = """ترجم الجملة التالية من الإنجليزية إلى الدارجة الجزائرية:

{sentence}

الترجمة: """

AR_TO_EN_TEMPLATE = """Translate the following Algerian Darija sentence to English:

{sentence}

Translation: """

def build_examples(row):
    darija = str(row["Arabic"]).strip()
    english = str(row["English"]).strip()
    return [
        {  # English -> Darija
            "prompt": EN_TO_AR_TEMPLATE.format(sentence=english),
            "completion": darija,
        },
        {  # Darija -> English
            "prompt": AR_TO_EN_TEMPLATE.format(sentence=darija),
            "completion": english,
        },
    ]

df = pd.read_csv("/kaggle/input/datasets/YOUR_USERNAME/YOUR_DATASET/translation_pairs.csv")  # id, Arabic, English
df = df.dropna(subset=["Arabic", "English"])

examples = []
for _, row in df.iterrows():
    examples.extend(build_examples(row))

pairs_df = pd.DataFrame(examples)
dataset = Dataset.from_pandas(pairs_df[["prompt", "completion"]])
splits = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = splits["train"]
eval_dataset = splits["test"]

pairs_df.shape

In [ ]:
print(train_dataset[0]["prompt"])
print("---")
print(train_dataset[0]["completion"])

In [ ]:
from trl import SFTConfig
output_dir = "/kaggle/working/gemma4-darija-en-translation-qlora"
training_args = SFTConfig(
    output_dir=output_dir,
    max_length=256,
    packing=False,                  # keep each prompt/completion pair as its own sequence
    dataset_num_proc=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # effective batch size = 16 examples/step
    learning_rate=1e-4,             # lower than Stage 1: adapting an already fine-tuned adapter
    lr_scheduler_type="cosine",
    num_train_epochs=3,             # translation pairs dataset is much smaller than Stage 1 corpus
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    completion_only_loss=True,      # only train the model to predict the completion, not the prompt
    report_to="none"
)

In [ ]:
# model already carries the Stage 1 LoRA adapter (is_trainable=True above).
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Stage 2 training finished & LoRA adapters saved!")

In [ ]:
# Load base model + Stage 2 LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map=device_map,
    dtype=compute_dtype
)
model_with_adapter = PeftModelForCausalLM.from_pretrained(base_model, output_dir)

# Darija -> English
prompt_ar = AR_TO_EN_TEMPLATE.format(sentence="حاب نتعلم البرمجة.")
inputs = tokenizer(prompt_ar, return_tensors="pt").to("cuda")
outputs = model_with_adapter.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# English -> Darija
prompt_en = EN_TO_AR_TEMPLATE.format(sentence="I want to learn programming.")
inputs = tokenizer(prompt_en, return_tensors="pt").to("cuda")
outputs = model_with_adapter.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))